## Install basic packages + reference genome


In [ ]:
# @title
!pip uninstall -y numpy --quiet
!pip install "numpy<2" --break-system-packages --quiet
!pip install ipywidgets IPython pyensembl --quiet
!apt-get install -y bowtie
!pip install pyranges biopython pysam --quiet

!wget https://genome-idx.s3.amazonaws.com/bt/GRCh38_noalt_as.zip
!unzip GRCh38_noalt_as.zip -d hg38_index

# GENCODE gene annotation
!wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.annotation.gtf.gz
!gunzip gencode.v44.annotation.gtf.gz

!pip install genomepy pyfaidx --quiet


In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import pyranges as pr
import subprocess
from Bio.Seq import Seq

BOWTIE_INDEX = "hg38_index/GRCh38_noalt_as/GRCh38_noalt_as"

!apt-get update && apt-get install -y bowtie2 -qq
import genomepy
print("Installing hg38 fasta (this takes a couple of minutes)...")
genomepy.install_genome("hg38", "UCSC", genomes_dir="./genomes")
from pyfaidx import Fasta

print("Loading GTF (this takes a minute)...")
gtf = pr.read_gtf("gencode.v44.annotation.gtf")
genes = gtf[gtf.Feature == "gene"]
del gtf  # free memory
print(f"Loaded {len(genes)} gene annotations.")

genomefasta = Fasta("./genomes/hg38/hg38.fa")

from pyensembl import Genome

genome = Genome(
    reference_name="GRCh38",
    annotation_name="ensembl115",
    gtf_path_or_url="https://ftp.ensembl.org/pub/release-115/gtf/homo_sapiens/Homo_sapiens.GRCh38.115.gtf.gz",
    transcript_fasta_paths_or_urls="https://ftp.ensembl.org/pub/release-115/fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz",
)
genome.download()
genome.index()
data = genome


## Helper Functions

In [ ]:
# @title
def reverse_complement(seq):
    tab = str.maketrans("ATCGatcg", "TAGCtagc")
    return seq.translate(tab)[::-1]

def getCandidateSeqs(exon_id, aso_len, min_saddle):
  if aso_len < 2*min_saddle:
    return "Target ASO length must be more than 2x as long as the minimum saddle on either side of the junction!"
  max_overhang = aso_len-min_saddle
  # ASO must overlap the intronic AND exonic side by min_saddle nt each
  exon = data.exon_by_id(exon_id)

  sequence5 = genomefasta['chr'+ exon.contig][exon.start- max_overhang-1 : exon.start+max_overhang-1]
  sequence3 = genomefasta['chr' + exon.contig][exon.end-max_overhang: exon.end+max_overhang]

  sequence5 = str(sequence5).split("\n", 1)[-1].replace("\n", "")
  sequence3 = str(sequence3).split("\n", 1)[-1].replace('\n', "")

  sequence5 = reverse_complement(sequence5) if exon.strand == '-' else sequence5
  sequence3 = reverse_complement(sequence3) if exon.strand == '-' else sequence3

  if not (exon.strand == '+'):
    sequence5, sequence3 = sequence3, sequence5

  sequence5 = sequence5[:max_overhang].lower() + sequence5[max_overhang:].upper()
  sequence3 = sequence3[:len(sequence3)-max_overhang].upper() + sequence3[len(sequence3)-max_overhang:].lower()

  # sequence5 - intron-exon junction on 5' end of coding strand
  # sequence3 - intron-exon junction on 3' end of coding strand
  return sequence5, sequence3

def getASOs(region, min_length, max_length, saddle):
  region = reverse_complement(region)  # ASO is the reverse complement of the coding strand
  ASOs = []
  for length in range(min_length, max_length+1):
    offset = length-(saddle*2)
    start = (len(region)/2)-saddle
    for i in range(int(start), int(start)-offset-1, -1):
      ASOs.append(region[i:i+length])
  return ASOs


In [ ]:
# @title
import re

def find_aso_off_targets(
    aso_seq: str,
    gene_annotations: pr.PyRanges,
    max_mismatches: int = 3,
    index_path: str = BOWTIE_INDEX,
) -> pd.DataFrame:
    MAX_EDIT_DIST = 5
    query = str(Seq(aso_seq.upper()).reverse_complement())

    min_score = -6 * MAX_EDIT_DIST - 5 * MAX_EDIT_DIST

    result = subprocess.run(
        [
            "bowtie2",
            "-x", index_path,
            "-c", query,
            "--end-to-end",
            "--very-sensitive",
            "-a",
            "--no-unal",
            "--score-min", f"L,{min_score},0",
            "--np", "0",
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(f"Bowtie2 error: {result.stderr.strip()}")

    rows = []
    for line in result.stdout.strip().split("\n"):
        if not line or line.startswith("@"):
            continue
        fields = line.split("\t")
        flag, chrom, pos, cigar = int(fields[1]), fields[2], int(fields[3]), fields[5]

        if chrom == "*":
            continue

        nm = [f for f in fields[11:] if f.startswith("NM:i:")]
        edit_dist = int(nm[0].split(":")[-1]) if nm else 0
        if edit_dist > MAX_EDIT_DIST:
            continue

        ref_len = sum(int(n) for n, op in re.findall(r"(\d+)([MDN=X])", cigar))

        rows.append({
            "Chromosome": chrom,
            "Start": pos - 1,
            "End": pos - 1 + ref_len,
            "Strand": "-" if flag & 16 else "+",
            "edit_distance": edit_dist,
        })

    if not rows:
        print(f"No genomic hits found for: {aso_seq}")
        return pd.DataFrame()

    hits = pr.PyRanges(pd.DataFrame(rows))
    total_hits = len(rows)

    off_targets = hits.join(gene_annotations)
    if off_targets.df.empty:
        print(f"{total_hits} genomic hits, but none overlap gene bodies.")
        return pd.DataFrame()

    result_df = (
        off_targets.df[["Chromosome", "Start", "End", "Strand",
                        "edit_distance", "gene_name", "gene_type"]]
        .sort_values("edit_distance")
        .reset_index(drop=True)
    )

    print(f"{total_hits} total genomic hits | "
          f"{len(result_df)} in gene bodies | ASO: {aso_seq}")
    return result_df


In [ ]:
# @title
import math

# Uses Sugimoto et al. (1988) to calculate nearest-neighbor dH and dS for RNA:DNA hybrids
NN_PARAMS = {
    "AA": (-11.5, -36.4), "AC": (-7.8, -21.6), "AG": (-7.0, -19.7), "AT": (-8.3, -23.9),
    "CA": (-10.4, -28.4), "CC": (-12.8, -31.9), "CG": (-16.3, -47.1), "CT": (-7.0, -19.7),
    "GA": (-7.8, -21.6), "GC": (-11.0, -28.5), "GG": (-12.8, -31.9), "GT": (-7.8, -21.6),
    "TA": (-7.8, -23.2), "TC": (-7.0, -19.7), "TG": (-10.4, -28.4), "TT": (-11.5, -36.4),
}
INIT_DH, INIT_DS = 1.9, -3.9
R = 1.987


def calc_hybrid_thermo(aso_seq, oligo_conc=0.25e-6):
    seq = aso_seq.upper().replace(" ", "").replace("U", "T")
    dH, dS = INIT_DH, INIT_DS
    for i in range(len(seq) - 1):
        d, s = NN_PARAMS[seq[i:i + 2]]
        dH += d
        dS += s

    Tm_C = (dH * 1000) / (dS + R * math.log(oligo_conc / 4)) - 273.15
    dG_37 = dH - 310.15 * dS / 1000

    return {
        "dna_seq": seq,
        "rna_seq": "".join({"A": "U", "T": "A", "G": "C", "C": "G"}[b] for b in seq),
        "length": len(seq),
        "dH": round(dH, 2),
        "dS": round(dS, 2),
        "dG_37": round(dG_37, 2),
        "Tm_C": round(Tm_C, 1),
    }


# Search for a gene and exon

In [ ]:
# @title
out = widgets.Output()
out2 = widgets.Output()

gene_name_entry = widgets.Text(description="Enter Ensembl gene name: ", placeholder='e.g. B2M', continuous_update=False)
id_display = widgets.Label(value="Ensemble ID: (Waiting for input)")

transcript_dropdown = widgets.Dropdown(
    description='Transcript:',
    options=[],
    style={'description_width': 'initial'}
)
exon_dropdown = widgets.Dropdown(description='Exon:')

length_slider = widgets.IntRangeSlider(description='ASO Length range:', min=15, max=30, value=(20, 25))
overlap_slider = widgets.IntSlider(description='Minimum Overlap:', min=1, max=15, value=1)

selected_exon_data = None
exons_list = []

def update_seqs(_=None):
    global seq5, seq3, aso_candidates
    out2.clear_output(wait=True)
    if selected_exon_data is None:
        return
    min_len, max_len = length_slider.value
    overlap = overlap_slider.value
    if min_len < 2*overlap:
      with out2:
        display('Minimum ASO length must be 2x larger than overlap!')
    else:
      seq5, seq3 = getCandidateSeqs(selected_exon_data.exon_id, max_len, overlap)
      # append + getASOs(seq3, min_len, max_len, overlap) to also include 3' junction ASOs
      aso_candidates = getASOs(seq5, min_len, max_len, overlap)
      with out2:
        display(f"5' Search {seq5}")
        display(f"3' Search {seq3}")
        display(f"Candidate windows: {[reverse_complement(aso) for aso in aso_candidates]}")
        display(f'{len(aso_candidates)} potential ASOs found!')

def update_exon_variable(change):
    global selected_exon_data
    exon_id = change['new']
    if exon_id:
        selected_exon_data = data.exon_by_id(exon_id)
        fixed_exons = [i for i, leng in exons_list if (leng+selected_exon_data.length) % 3 == 0]
        with out:
            clear_output()
            print(f"Exon {exon_id} (length {selected_exon_data.length}). "
                  f"Skipping it alongside these exons stays in-frame: {fixed_exons}")
    else:
        selected_exon_data = None
    update_seqs()

def update_exons(change):
    global exons_list
    transcript_id = change['new']
    if transcript_id:
        try:
            exon_ids = data.exon_ids_of_transcript_id(transcript_id)
            exons_list = [(i+1, data.exon_by_id(eid).length) for i, eid in enumerate(exon_ids)]
            exon_options = [(f"Exon {i+1} ({eid})", eid) for i, eid in enumerate(exon_ids)]
            exon_dropdown.options = exon_options
        except Exception:
            exon_dropdown.options = [("No exons found", None)]
    else:
        exon_dropdown.options = []

def update_transcripts(change):
    name = change['new'].strip().upper()
    if name:
        try:
            genes = data.genes_by_name(name)
            if not genes:
                id_display.value = f"Ensemble ID: No gene found for {name}"
                transcript_dropdown.options = []
                return
            gene_id = genes[0].gene_id
            id_display.value = f'Ensemble ID: {gene_id}'
            transcript_ids = data.transcript_ids_of_gene_id(gene_id)
            transcript_options = []
            for t_id in transcript_ids:
                n_exons = len(data.exon_ids_of_transcript_id(t_id))
                t_obj = data.transcript_by_id(t_id)
                transcript_options.append((f'{t_obj.transcript_name} ({n_exons} exons)', t_id))
            transcript_dropdown.options = transcript_options
            if transcript_options:
                update_exons({'new': transcript_options[0][1]})
        except Exception as e:
            id_display.value = f"Error: {str(e)}"
            transcript_dropdown.options = []
    else:
        id_display.value = 'Ensembl ID: (Waiting for input)'
        transcript_dropdown.options = []

gene_name_entry.observe(update_transcripts, names='value')
transcript_dropdown.observe(update_exons, names='value')
exon_dropdown.observe(update_exon_variable, names='value')
length_slider.observe(update_seqs, 'value')
overlap_slider.observe(update_seqs, 'value')

ui = widgets.VBox([
    gene_name_entry, id_display,
    transcript_dropdown, exon_dropdown,
    length_slider, overlap_slider,
    out, out2
])
display(ui)


In [ ]:
# @title
df = pd.DataFrame()
for aso in aso_candidates:
  mismatch = find_aso_off_targets(aso, genes)
  thermals = calc_hybrid_thermo(aso)

  length = len(aso)
  gc = (aso.count('G') + aso.count('C'))/length*100
  new_row = pd.DataFrame({
      "Sequence": [aso],
      "Length": [length],
      "GC %": [gc],
      "MM0": len(mismatch[mismatch["edit_distance"] == 0])-1,
      "MM1": len(mismatch[mismatch["edit_distance"] == 1]),
      "MM2": len(mismatch[mismatch["edit_distance"] == 2]),
      "MM3": len(mismatch[mismatch["edit_distance"] == 3]),
      "MM0 genes": str(mismatch[mismatch["edit_distance"] == 0]["gene_name"].values),
      "MM1 genes": str(mismatch[mismatch["edit_distance"] == 1]["gene_name"].values),
      "dG": thermals["dG_37"],
      "dH": thermals["dH"],
      "dS": thermals["dS"],
      "Tm": thermals["Tm_C"],
  })
  df = pd.concat([df, new_row], ignore_index=True)
df


## Show best candidates

In [ ]:
filtereddf = df[(df["GC %"] >= 40.0) & (df["GC %"] <= 60.0)]
filtereddf = filtereddf.sort_values(by="GC %", ascending=False)
filtereddf = filtereddf.sort_values(by="MM1", ascending=True)
filtereddf = filtereddf.sort_values(by="MM0", ascending=True)
filtereddf


# Get detailed look at a candidate's mismatches


In [ ]:
# @title Find Off-Target Binding Sites for an ASO
def update_df(change):
    with mismatch_out:
      clear_output()
      try:
        mismatch = find_aso_off_targets(change['new'], genes)
        display(mismatch)

        print(f"\n--- Selected ASO perfect-match thermodynamics ---")
        result = calc_hybrid_thermo(change['new'])
        print(f"ASO (DNA, 5'→3'):   {result['dna_seq']}")
        print(f"RNA target (3'→5'): {result['rna_seq']}")
        print(f"Length:     {result['length']} nt")
        print(f"dH:        {result['dH']:+.2f} kcal/mol")
        print(f"dS:        {result['dS']:+.2f} cal/(mol·K)")
        print(f"dG(37°C):  {result['dG_37']:+.2f} kcal/mol")
        print(f"Tm:        {result['Tm_C']:.1f} °C")

        print(f"\n--- Casimersen perfect-match thermodynamics (reference) ---")
        casi = calc_hybrid_thermo("CAATGCCATCCTGGAGTTCCTG")
        print(f"ASO (DNA, 5'→3'):   {casi['dna_seq']}")
        print(f"RNA target (3'→5'): {casi['rna_seq']}")
        print(f"Length:     {casi['length']} nt")
        print(f"dH:        {casi['dH']:+.2f} kcal/mol")
        print(f"dS:        {casi['dS']:+.2f} cal/(mol·K)")
        print(f"dG(37°C):  {casi['dG_37']:+.2f} kcal/mol")
        print(f"Tm:        {casi['Tm_C']:.1f} °C")
      except Exception:
        print('Enter a valid ASO sequence.')

mismatch_out = widgets.Output()
aso_textbox = widgets.Text(description='ASO: ', placeholder='Enter ASO sequence (e.g., TTTCTTCACGGACAGTGT)')
aso_textbox.observe(update_df, names='value')
display(aso_textbox, mismatch_out)
